# LASSO Regression

In [ ]:
from analyses.linear_regression.multiple_linear_regression import get_specific_behavioral_matrix
from analyses.enums.monkey_names import get_monkeys_by_default_order
import pandas as pd
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
aff = get_specific_behavioral_matrix("affiliation")
sub = get_specific_behavioral_matrix("submission")
ago = get_specific_behavioral_matrix("agonism")

monkey_group = "Zombies"
behavior_type = "Affiliation"
monkeys = get_monkeys_by_default_order(monkey_group)

In [ ]:
if behavior_type == "Affiliation":
    X = aff
elif behavior_type=="Submission":
    X = sub
elif behavior_type=="Agonism":
    X = ago
else:
    raise ValueError("Behavior Type should be one of 'affiliation', 'submission', 'agonism' ")
X_df = pd.DataFrame(X, index=monkeys, columns=[f'Feature{i+1}' for i in range(X.shape[1])])
X_df_clean = X_df.drop(index="81G")

In [ ]:
from analyses.spike_rate import compute_mean_spike_rate_for_windows
from analyses.enums.monkey_names import get_monkeys_by_default_order
import pandas as pd
monkey_group = "Zombies"
subject_monkey_index = 6
monkey_list = get_monkeys_by_default_order(monkey_group)
sig_windows = pd.read_pickle(
        f'/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_cache/{monkey_group}_significant_windows_pANOVAorGLM_passed.pkl')
sig_windows = sig_windows[sig_windows['PermANOVA_p-value']<0.05]
ed_sig_windows = pd.read_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/Ed and ANOVA/used_for_R01/R01_ed_list_duplicate_removed_without_full_window.pkl')

# Which Y to use for the regression
mean_spike_rate_windows = compute_mean_spike_rate_for_windows(sig_windows)

In [ ]:
spike_rate = mean_spike_rate_windows[mean_spike_rate_windows['MonkeyGroup']=='Zombies']
spike_rate = spike_rate[spike_rate['MonkeyName']!='NewMonkey']
# spike_rate = spike_rate[spike_rate['MonkeyName']!='7124']
# spike_rate['MonkeyName'].unique()

In [ ]:
from analyses.linear_regression.multiple_linear_regression import run_lasso_regression

results_df = run_lasso_regression(spike_rate, X_df_clean)
results_df.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract the coefficients column as a new DataFrame
coeff_df = pd.DataFrame(results_df['coefficients'].tolist(),
                        index=results_df['NeuronWindowID']) 

filtered_monkey_list = [m for m in monkey_list if m not in ["81G"]]
coeff_df.columns = monkey_list
sparse_df = coeff_df.replace(0, float('nan')) 
plt.figure(figsize=(12, 10))
ax = sns.heatmap(sparse_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)



# Adjust label fonts AFTER the heatmap has rendered
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# Fix for misalignment: force re-draw with tight layout
plt.tight_layout()
plt.title("Heatmap of Regression Coefficients per Neuron", fontsize=14)
plt.xlabel(f"{behavior_type} From")
plt.ylabel("Neuron ID")
plt.show()

In [ ]:
# Sparse Heatmap for R-sq > 0.6 cells
sig_results_df = results_df[results_df['R_squared']>0.6]
sig_coeff_df = pd.DataFrame(sig_results_df['coefficients'].tolist(),
                        index=sig_results_df['NeuronWindowID']) 
sparse_df = sig_coeff_df.replace(0, float('nan'))  # Optional: or use a threshold like < 1e-5
sparse_df.columns = monkey_list
plt.figure(figsize=(12, 10))
ax = sns.heatmap(sparse_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)

ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
plt.tight_layout()
plt.title("Heatmap of Lasso Coeff (Non-zero only) for R-sq > 0.6", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()


In [ ]:
selection_counts = (coeff_df != 0).sum(axis=0).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=selection_counts.values, y=selection_counts.index)
plt.title("Number of Neurons Selecting Each Feature (Non-zero Coefficients)")
plt.xlabel("Count")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.6
n_sig = (results_df['R_squared'] > threshold).sum()
n_total = len(results_df)

print(f"{n_sig} out of {n_total} windows had R² > {threshold:.2f} ({n_sig / n_total:.1%})")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.figure(figsize=(6, 4))
plt.hist(results_df['R_squared'], bins=30, color='gray', edgecolor='black')
plt.xlabel('R-squared')
plt.ylabel('Number of Windows')
plt.title('Distribution of R² across Neuron × Window')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def analyze_feature_selection(results_df, feature_names):
    """Analyze which features were selected by LASSO"""
    
    # For each regression result, find non-zero coefficients
    selected_features = []
    
    for idx, row in results_df.iterrows():
        coeffs = np.array(row['coefficients'])
        non_zero_idx = np.where(np.abs(coeffs) > 1e-10)[0]  # Account for numerical precision
        
        selected_features.append({
            'NeuronWindowID': row['NeuronWindowID'],
            'selected_features': [feature_names[i] for i in non_zero_idx],
            'selected_coeffs': coeffs[non_zero_idx].tolist(),
            'n_features_selected': len(non_zero_idx)
        })
    
    return pd.DataFrame(selected_features)
selected_feat = analyze_feature_selection(results_df, monkeys)
selected_feat.head()

In [ ]:
def feature_importance_analysis(selection_results, all_feature_names):
    """Analyze feature importance across all neurons/windows"""
    
    # Count how often each feature is selected
    feature_counts = {}
    feature_coeff_magnitudes = {}
    
    for _, row in selection_results.iterrows():
        for feature, coeff in zip(row['selected_features'], row['selected_coeffs']):
            feature_counts[feature] = feature_counts.get(feature, 0) + 1
            if feature not in feature_coeff_magnitudes:
                feature_coeff_magnitudes[feature] = []
            feature_coeff_magnitudes[feature].append(abs(coeff))
    
    # Create summary
    feature_summary = []
    total_regressions = len(selection_results)
    
    for feature in all_feature_names:
        count = feature_counts.get(feature, 0)
        avg_magnitude = np.mean(feature_coeff_magnitudes.get(feature, [0]))
        
        feature_summary.append({
            'feature': feature,
            'selection_frequency': count / total_regressions,
            'selection_count': count,
            'avg_coeff_magnitude': avg_magnitude,
            'importance_score': count * avg_magnitude  # Combined metric
        })
    
    return pd.DataFrame(feature_summary).sort_values('importance_score', ascending=False)
feat_summary = feature_importance_analysis(selected_feat, monkeys)
feat_summary.head()

In [ ]:
# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Selection frequency
ax1.barh(range(len(feat_summary)), feat_summary['selection_frequency'])
ax1.set_yticks(range(len(feat_summary)))
ax1.set_yticklabels(feat_summary['feature'])
ax1.set_xlabel('Selection Frequency')
ax1.set_title('Feature Selection Frequency')

# Average coefficient magnitude
ax2.barh(range(len(feat_summary)), feat_summary['avg_coeff_magnitude'])
ax2.set_yticks(range(len(feat_summary)))
ax2.set_yticklabels(feat_summary['feature'])
ax2.set_xlabel('Average |Coefficient|')
ax2.set_title('Average Coefficient Magnitude')

plt.tight_layout()
plt.show()


In [ ]:
from analyses.linear_regression.multiple_linear_regression import run_multiple_regression
run_multiple_regression(spike_rate, X_df_clean, 'lassocv')

# RIDGE Regression

In [ ]:
from analyses.linear_regression.multiple_linear_regression import run_ridge_regression

ridge_results_df = run_ridge_regression(spike_rate, X_df_clean)
ridge_results_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Extract the coefficients column as a new DataFrame
coeff_df = pd.DataFrame(ridge_results_df['coefficients'].tolist(),
                        index=ridge_results_df['NeuronWindowID']) 

filtered_monkey_list = [m for m in monkey_list if m not in ["81G"]]
coeff_df.columns = monkey_list

plt.figure(figsize=(12, 10))
ax = sns.heatmap(coeff_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)



# Adjust label fonts AFTER the heatmap has rendered
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# Fix for misalignment: force re-draw with tight layout
plt.tight_layout()
plt.title("Heatmap of Regression Coefficients per Neuron", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()

# ELASTIC NET Regression

In [ ]:
from analyses.linear_regression.multiple_linear_regression import run_elasticnet_regression

elasticnet_results_df = run_elasticnet_regression(spike_rate, X_df_clean)
print(elasticnet_results_df)

In [ ]:
# Extract the coefficients column as a new DataFrame
coeff_df = pd.DataFrame(elasticnet_results_df['coefficients'].tolist(),
                        index=elasticnet_results_df['NeuronWindowID']) 

filtered_monkey_list = [m for m in monkey_list if m not in ["81G"]]
coeff_df.columns = monkey_list

plt.figure(figsize=(12, 10))
ax = sns.heatmap(coeff_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)



# Adjust label fonts AFTER the heatmap has rendered
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# Fix for misalignment: force re-draw with tight layout
plt.tight_layout()
plt.title("Heatmap of Regression Coefficients per Neuron", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()